# Colab Setup — Run This First Every Session

**Runtime:** Runtime → Change runtime type → T4 GPU

This notebook:
1. Mounts Google Drive
2. Installs all packages
3. Verifies GPU
4. Creates the project folder structure in Drive

**Run once per Colab session** (packages reset when session ends).

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All project files live here — persistent across sessions
PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
print(f'Project directory: {PROJECT_DIR}')

## Step 2 — Install packages

In [ ]:
# Colab already has torch/torchvision — just install the extras
!pip install -q ultralytics albumentations supervision timm roboflow
print('Packages installed.')

## Step 3 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Step 4 — Create folder structure in Drive

In [ ]:
import os

dirs = [
    'data/annotated/images/train',
    'data/annotated/images/val',
    'data/annotated/images/test',
    'data/annotated/images/calibration',
    'data/annotated/labels/train',
    'data/annotated/labels/val',
    'data/annotated/labels/test',
    'data/annotated/labels/calibration',
    'data/corrupted',
    'data/raw',
    'models/baseline',
    'models/robust',
    'results/figures',
    'notes',
]

for d in dirs:
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)

print('✅ Folder structure created in', PROJECT_DIR)

## Step 5 — Write robot_parts.yaml

This is the dataset config Ultralytics reads for training.

In [ ]:
yaml_content = f"""path: {PROJECT_DIR}/data/annotated
train: images/train
val: images/val
test: images/test

nc: 5
names: ['arm', 'leg', 'torso', 'head', 'sensor']
"""

yaml_path = os.path.join(PROJECT_DIR, 'robot_parts.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f'Written: {yaml_path}')
print(yaml_content)

## Step 6 — Symlink Drive project into /content for faster I/O

Ultralytics reads images from disk during training. Drive I/O is slow,
so we copy data to local `/content/` at the start of each training session.

In [ ]:
# Copy annotated data to local disk for fast training I/O
# Run this cell right before training (after dataset is ready in Drive)
import shutil

LOCAL_DATA = '/content/robot_data'
if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)

shutil.copytree(
    os.path.join(PROJECT_DIR, 'data/annotated'),
    LOCAL_DATA
)
print(f'Data copied to {LOCAL_DATA}')

# Count images
for split in ['train', 'val', 'test']:
    n = len([f for f in os.listdir(f'{LOCAL_DATA}/images/{split}')
             if f.endswith(('.jpg','.png','.jpeg'))])
    print(f'  {split}: {n} images')

---
Setup complete. Open the next notebook for your current phase.